# Access to Nature

[dataset](https://www.gov.uk/government/statistics/access-to-green-and-blue-space-in-england/access-to-green-and-blue-space-in-england)

## context:

the government made a committment in the 2025 [environmental improvement plan](https://www.gov.uk/government/publications/environmental-improvement-plan-2025/environmental-improvement-plan-eip-2025#chapter-5-access-to-nature) to ensure all households have access to green/blue space within a 15 minute walk.

This is called the 15 minute committment.
Hoseholds meet the 15 minute committment depending on the proximity and size of the accessible greenspace.
There are 3 classifications overall, a hectare is a square of 100m. About the size of a football pitch, or the area inside a 400m running track.
- doorstep - a small area of greenspace (>0.5 hectares) within 200 meters
- local - a medium area of greenspace (>2 hectares) within 300 meters
- neighborhood - a large area of greenspace (>10 hectares open space) or a walkable path > 500m of through private greenspace or bluespace (such as beside a golfcourse or along a river).

It forms part of the [Local Nature Recovery Strategies](https://www.gov.uk/government/publications/local-nature-recovery-strategies/local-nature-recovery-strategies) for which Local Government (Strategic Authorities) will be respnsible to consider during planning.

Following LGR surrey will be a Strategic (Mayoral) Authority.

In [89]:
import polars as pl
import fastexcel
import fsspec
import polars.selectors as cs
from polars.testing import assert_frame_equal, assert_frame_not_equal
import itertools
from jsna_climate_and_environment.geography.places import SURREY_EAST, SURREY_DISTRICTS

In [9]:
with fsspec.open(
    "https://assets.publishing.service.gov.uk/media/69a184aef534e7e99adaeab4/Access_to_green_and_blue_space_England_data_table.ods"
) as file:
    reader = fastexcel.read_excel(file.read())
reader.sheet_names

['Cover_sheet', 'Table_of_contents', 'Notes', '1', '2', '3']

In [20]:
reader.load_sheet(3, n_rows=0).to_polars().columns[0]

'Percentage of households in each Output Area with access to green and blue space in England, under the 15-minute commitment and neighbourhood, local, and doorstep standards'

In [23]:
green_blue_df = reader.load_sheet(3, header_row=4).to_polars()
green_blue_df.columns

['OA21CD',
 'LSOA21CD',
 'LSOA21NM',
 'MSOA21CD',
 'MSOA21NM',
 'LAD25CD',
 'LAD25NM',
 'total_uprn',
 'uprn_in_commitment',
 'percentage_in_commitment',
 'uprn_in_doorstep',
 'percentage_in_doorstep',
 'uprn_in_local',
 'percentage_in_local',
 'uprn_in_neighbourhood',
 'percentage_in_neighbourhood',
 'uprn_in_doorstep_and_local',
 'percentage_in_doorstep_and_local',
 'uprn_in_local_and_neighbourhood',
 'percentage_in_local_and_neighbourhood',
 'uprn_in_doorstep_and_neighbourhood',
 'percentage_in_doorstep_and_neighbourhood',
 'uprn_in_doorstep_and_local_and_neighbourhood',
 'percentage_in_doorstep_and_local_and_neighbourhood',
 'urban_rural_flag']

In [82]:
blue_df = reader.load_sheet(4, header_row=4).to_polars()
blue_df.columns

['OA21CD',
 'LSOA21CD',
 'LSOA21NM',
 'MSOA21CD',
 'MSOA21NM',
 'LAD25CD',
 'LAD25NM',
 'total_uprn',
 'uprn_in_commitment',
 'percentage_in_commitment',
 'urban_rural_flag']

In [83]:
green_df = reader.load_sheet(5, header_row=4).to_polars()
green_df.columns

['OA21CD',
 'LSOA21CD',
 'LSOA21NM',
 'MSOA21CD',
 'MSOA21NM',
 'LAD25CD',
 'LAD25NM',
 'total_uprn',
 'uprn_in_commitment',
 'percentage_in_commitment',
 'uprn_in_doorstep',
 'percentage_in_doorstep',
 'uprn_in_local',
 'percentage_in_local',
 'uprn_in_neighbourhood',
 'percentage_in_neighbourhood',
 'uprn_in_doorstep_and_local',
 'percentage_in_doorstep_and_local',
 'uprn_in_local_and_neighbourhood',
 'percentage_in_local_and_neighbourhood',
 'uprn_in_doorstep_and_neighbourhood',
 'percentage_in_doorstep_and_neighbourhood',
 'uprn_in_doorstep_and_local_and_neighbourhood',
 'percentage_in_doorstep_and_local_and_neighbourhood',
 'urban_rural_flag']

In [87]:
assert_frame_equal(
    green_blue_df.select(~cs.contains("commitment", "neighbourhood")),
    green_df.select(~cs.contains("commitment", "neighbourhood")),
    check_row_order=False
)

In [90]:
assert_frame_not_equal(
    green_blue_df,
    green_df,
    check_row_order=False
)

In [102]:
def percent_calc(numerator: pl.Expr, denominator: pl.Expr) -> pl.Expr:
    return ((numerator / denominator) * 100).round(2, mode="half_away_from_zero")

def unpivot_numerators(df: pl.DataFrame, measure_name: str) -> pl.DataFrame:
    return df.unpivot(
        on=cs.starts_with("uprn_in") & ~cs.contains("_and_"),
        index=cs.ends_with("CD") | cs.by_name("total_uprn", "urban_rural_flag"),
        value_name="numerator",
        variable_name="indicator"
    ).with_columns(
        denominator="total_uprn",
        indicator=pl.col("indicator").str.strip_prefix("uprn_in_"),
        measure=pl.lit(measure_name),
        value=percent_calc(pl.col("numerator"), pl.col("total_uprn"))
    )

long_df = pl.concat([
    unpivot_numerators(green_blue_df.select(~cs.contains("local", "doorstep")), "greenspace or bluespace"),
    unpivot_numerators(green_df, "greenspace only"),
    unpivot_numerators(blue_df, "bluespace only")
])
long_df

OA21CD,LSOA21CD,MSOA21CD,LAD25CD,total_uprn,urban_rural_flag,indicator,numerator,denominator,measure,value
str,str,str,str,f64,str,str,f64,f64,str,f64
"""E00095134""","""E01018844""","""E02003906""","""E06000052""",131.0,"""Rural""","""commitment""",131.0,131.0,"""greenspace or bluespace""",100.0
"""E00095360""","""E01018882""","""E02003928""","""E06000052""",136.0,"""Urban""","""commitment""",135.0,136.0,"""greenspace or bluespace""",99.26
"""E00095520""","""E01018916""","""E02003921""","""E06000052""",196.0,"""Urban""","""commitment""",192.0,196.0,"""greenspace or bluespace""",97.96
"""E00096408""","""E01019077""","""E02006781""","""E06000053""",84.0,"""Rural""","""commitment""",66.0,84.0,"""greenspace or bluespace""",78.57
"""E00182783""","""E01018914""","""E02003920""","""E06000052""",81.0,"""Urban""","""commitment""",54.0,81.0,"""greenspace or bluespace""",66.67
…,…,…,…,…,…,…,…,…,…,…
"""E00174127""","""E01012119""","""E02002532""","""E06000003""",135.0,"""Urban""","""commitment""",22.0,135.0,"""bluespace only""",16.3
"""E00061092""","""E01033471""","""E02002523""","""E06000003""",121.0,"""Urban""","""commitment""",0.0,121.0,"""bluespace only""",0.0
"""E00061416""","""E01012177""","""E02002518""","""E06000003""",125.0,"""Urban""","""commitment""",15.0,125.0,"""bluespace only""",12.0


In [105]:
percentage=((pl.col("numerator") / pl.col("denominator")) * 100).round(2, mode="half_away_from_zero")

england_val = long_df.group_by("indicator", "measure").agg(
    england_numerator=pl.sum("numerator"),
    england_denominator=pl.sum("denominator"),
    england_value=percent_calc(pl.sum("numerator"), pl.sum("denominator"))
).sort("indicator", "measure")
england_val

indicator,measure,england_numerator,england_denominator,england_value
str,str,f64,f64,f64
"""commitment""","""bluespace only""",9.664736e6,2.5974871e7,37.21
"""commitment""","""greenspace only""",1.8331079e7,2.5974871e7,70.57
"""commitment""","""greenspace or bluespace""",2.0794736e7,2.5974871e7,80.06
"""doorstep""","""greenspace only""",4.139729e6,2.5974871e7,15.94
"""local""","""greenspace only""",3.531268e6,2.5974871e7,13.59
"""neighbourhood""","""greenspace only""",1.6755199e7,2.5974871e7,64.51
"""neighbourhood""","""greenspace or bluespace""",1.9810702e7,2.5974871e7,76.27


In [106]:
surrey_df = long_df.filter(pl.col("LAD25CD").is_in(SURREY_DISTRICTS))
surrey_val = surrey_df.group_by("indicator", "measure").agg(
    surrey_numerator=pl.sum("numerator"),
    surrey_denominator=pl.sum("denominator"),
    surrey_value=percent_calc(pl.sum("numerator"), pl.sum("denominator"))
).sort("indicator", "measure")
surrey_val

indicator,measure,surrey_numerator,surrey_denominator,surrey_value
str,str,f64,f64,f64
"""commitment""","""bluespace only""",183310.0,525717.0,34.87
"""commitment""","""greenspace only""",383137.0,525717.0,72.88
"""commitment""","""greenspace or bluespace""",429967.0,525717.0,81.79
"""doorstep""","""greenspace only""",98440.0,525717.0,18.72
"""local""","""greenspace only""",92603.0,525717.0,17.61
"""neighbourhood""","""greenspace only""",361021.0,525717.0,68.67
"""neighbourhood""","""greenspace or bluespace""",417207.0,525717.0,79.36


In [109]:
description = pl.concat_str(pl.col("indicator"), pl.col("measure"), separator=" ").replace_strict({
    "commitment greenspace or bluespace": "Households that meet any of the government's commitments to access",
    "commitment bluespace only": "Households that are within 1km walk of more than 500m walkable bluespace",
    "commitment greenspace only": "Households that meet any of the government's commitments to access excluding bluespace",
    "doorstep greenspace only": "Households that are within 200m walk of more than 50m walkable greenspace",
    "local greenspace only": "Households that are within 300m walk of more than 200m walkable greenspace",
    "neighbourhood greenspace only": "Households that are within 1km walk of more than 500m walkable greenspace",
    "neighbourhood greenspace or bluespace": "Households that are within 1km walk of more than 500m walkable bluespace or greenspace",
})


meta = surrey_val.join(england_val, on=["indicator", "measure"]).with_columns(
    description=description
)
meta

indicator,measure,surrey_numerator,surrey_denominator,surrey_value,england_numerator,england_denominator,england_value,description
str,str,f64,f64,f64,f64,f64,f64,str
"""commitment""","""bluespace only""",183310.0,525717.0,34.87,9.664736e6,2.5974871e7,37.21,"""Households that are within 1km…"
"""commitment""","""greenspace only""",383137.0,525717.0,72.88,1.8331079e7,2.5974871e7,70.57,"""Households that meet any of th…"
"""commitment""","""greenspace or bluespace""",429967.0,525717.0,81.79,2.0794736e7,2.5974871e7,80.06,"""Households that meet any of th…"
"""doorstep""","""greenspace only""",98440.0,525717.0,18.72,4.139729e6,2.5974871e7,15.94,"""Households that are within 200…"
"""local""","""greenspace only""",92603.0,525717.0,17.61,3.531268e6,2.5974871e7,13.59,"""Households that are within 300…"
"""neighbourhood""","""greenspace only""",361021.0,525717.0,68.67,1.6755199e7,2.5974871e7,64.51,"""Households that are within 1km…"
"""neighbourhood""","""greenspace or bluespace""",417207.0,525717.0,79.36,1.9810702e7,2.5974871e7,76.27,"""Households that are within 1km…"


In [114]:
from jsna_climate_and_environment.config import OUTPUT_DIR
meta.write_csv(OUTPUT_DIR / "access_to_nature_meta.csv")

# Data Format
data is aggregated at Output Area where each household by uprn calculates a distance from a known node.
This calculation travels the shortest straight line distance to a link and at the intersection the remaining distance to the closest node.
Each node has a shortest distance from some form of greenspace.

the distances for each uprn are measured against a threshold and classified. The number within the threshold is calculated in order to get a proportion value.

In [112]:
districts = surrey_df.group_by("indicator", "measure", "LAD25CD").agg(
    pl.sum("numerator"),
    pl.sum("denominator"),
    value=percent_calc(pl.sum("numerator"), pl.sum("denominator"))
).sort("indicator")
districts

indicator,measure,LAD25CD,numerator,denominator,value
str,str,str,f64,f64,f64
"""commitment""","""greenspace only""","""E07000211""",49944.0,64609.0,77.3
"""commitment""","""greenspace or bluespace""","""E07000208""",25511.0,33386.0,76.41
"""commitment""","""bluespace only""","""E07000216""",22003.0,57775.0,38.08
"""commitment""","""bluespace only""","""E07000211""",10060.0,64609.0,15.57
"""commitment""","""greenspace only""","""E07000217""",36032.0,44664.0,80.67
…,…,…,…,…,…
"""neighbourhood""","""greenspace only""","""E07000217""",35086.0,44664.0,78.56
"""neighbourhood""","""greenspace only""","""E07000207""",31952.0,60573.0,52.75
"""neighbourhood""","""greenspace only""","""E07000212""",24950.0,39614.0,62.98


In [115]:
districts.write_csv(OUTPUT_DIR / "access_to_nature_district.csv")

In [118]:
lsoas = surrey_df.group_by("indicator", "measure", "LSOA21CD").agg(
    pl.col("LAD25CD").unique().item(),
    pl.sum("numerator"),
    pl.sum("denominator"),
    value=percent_calc(pl.sum("numerator"), pl.sum("denominator"))
).sort("indicator", "value")
lsoas

indicator,measure,LSOA21CD,LAD25CD,numerator,denominator,value
str,str,str,str,f64,f64,f64
"""commitment""","""bluespace only""","""E01030518""","""E07000210""",0.0,557.0,0.0
"""commitment""","""bluespace only""","""E01030464""","""E07000209""",0.0,649.0,0.0
"""commitment""","""bluespace only""","""E01030508""","""E07000210""",0.0,558.0,0.0
"""commitment""","""bluespace only""","""E01030599""","""E07000211""",0.0,744.0,0.0
"""commitment""","""bluespace only""","""E01030343""","""E07000207""",0.0,716.0,0.0
…,…,…,…,…,…,…
"""neighbourhood""","""greenspace or bluespace""","""E01030513""","""E07000210""",530.0,530.0,100.0
"""neighbourhood""","""greenspace or bluespace""","""E01030312""","""E07000207""",594.0,594.0,100.0
"""neighbourhood""","""greenspace or bluespace""","""E01030593""","""E07000211""",646.0,646.0,100.0


In [119]:
lsoas.write_csv(OUTPUT_DIR / "access_to_nature_lsoa.csv")